In [ ]:
!pip -q install requests ipywidgets bencodepy

# Archive Torrents — download from Archive.org to Google Drive

Download ISO images from Archive.org pages using their torrent when available.
If an item has no torrent, or the torrent fails, download the same files over HTTPS. Paste inputs into an empty widget.

---

## Source
- **Site / tool:** https://archive.org / https://aria2.github.io/
- **How to find input:** paste `/download/<identifier>` links, `/details/<identifier>` links, or identifiers, one per line.
- Requested examples (copy them into the widget):
  ```text
  https://archive.org/download/red-star-en
  https://archive.org/download/NeoKylinLinuxDesktop6.0I386B04520141201
  ```
  When this notebook was created, Red Star provided a torrent and NeoKylin provided only an ISO over HTTPS.

## What it does
1. Fetches metadata and selects ISO images (or original files if you disable the option).
2. Shows file sizes and the download method, mounts Drive, and checks temporary disk space.
3. Downloads with aria2, verifies Archive.org SHA-1 checksums, and falls back to HTTPS if the torrent fails.
4. Copies only complete, verified files to Drive in the final cell.

## How to use
1. Run the first cell to install libraries. Use a CPU runtime; no GPU is required.
2. Run the widget cell and paste both links, one per line. Keep “ISO only” enabled for these systems.
3. Run setup and authorize Google Drive; it installs aria2 if missing.
4. Run the functions cell, then the download cell. You can select HTTPS if BitTorrent stalls.
5. Run the final cell to save to Drive, even if some items failed.
6. If you interrupt a download, rerun the download cell within the same runtime.

## Known limitations
- Requires free space on both the Colab disk and Drive; the Drive quota may cause copy errors.
- Temporary files are lost if Colab deletes the runtime. Files are saved only when you run the final cell.
- Availability depends on Archive.org, peers, and the Colab network. Torrents are limited to 15 minutes per item before trying HTTPS.
- Does not seed after completion; BitTorrent may upload pieces to other peers during downloads.
- “ISO only” may include several variants: Red Star contains two ISO images. Disable it for other original formats.
- Does not support private items requiring login, magnet links, or external torrents.
- Rerunning the copy replaces files with the same name. The notebook never executes downloaded ISO images.

## Output
```text
MyDrive/CollabMedia/downloads/archive-torrents/
└── <identifier>/
    └── <file.iso>
```
Temporary files: `/content/downloads/archive-torrents/`.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

urls_box = widgets.Textarea(value="", placeholder="One Archive.org link or identifier per line",
                            layout=widgets.Layout(width="100%", height="160px"))
iso_only = widgets.Checkbox(value=True, description="ISO only")
method_box = widgets.Dropdown(options=[("Torrent with HTTPS fallback", "auto"),
                                      ("HTTPS only", "https")], value="auto", description="Method:")
display(widgets.Label("Archive.org items:"), urls_box, iso_only, method_box)

In [ ]:
import hashlib
import os
import re
import shutil
import subprocess
from pathlib import Path
from urllib.parse import quote, unquote, urlparse
import bencodepy
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google.colab import drive

inputs = list(dict.fromkeys(x.strip() for x in urls_box.value.splitlines() if x.strip()))
if not inputs:
    raise ValueError("Paste at least one link into the widget before continuing.")
ONLY_ISO, METHOD = iso_only.value, method_box.value
WORK = Path("/content/downloads/archive-torrents")
DEST = Path("/content/drive/MyDrive/CollabMedia/downloads/archive-torrents")
WORK.mkdir(parents=True, exist_ok=True)
if not shutil.which("aria2c"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], check=True)
drive.mount("/content/drive")
http = requests.Session()
http.headers["User-Agent"] = "archive-colab-downloader/1.0"
http.mount("https://", HTTPAdapter(max_retries=Retry(total=4, backoff_factor=2,
            status_forcelist=[429, 500, 502, 503, 504])))

In [ ]:
def identifier(value):
    if "://" in value:
        u = urlparse(value)
        parts = u.path.strip("/").split("/")
        if u.scheme != "https" or u.hostname not in {"archive.org", "www.archive.org"}:
            raise ValueError("Use an archive.org HTTPS URL.")
        if len(parts) != 2 or parts[0] not in {"details", "download"}:
            raise ValueError("Use the item page rather than a direct file link.")
        value = unquote(parts[1])
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", value):
        raise ValueError(f"Invalid identifier: {value!r}")
    return value


def safe_path(base, name):
    path = base / name
    if not name or "\\" in name or Path(name).is_absolute():
        raise ValueError(f"Invalid path: {name!r}")
    if not path.resolve().is_relative_to(base.resolve()) or path.resolve() == base.resolve():
        raise ValueError(f"Path outside destination: {name!r}")
    return path


def file_url(item, name):
    return f"https://archive.org/download/{quote(item, safe='')}/{quote(name, safe='/')}"


def get_plan(item):
    response = http.get(f"https://archive.org/metadata/{item}", timeout=(15, 90))
    response.raise_for_status()
    data = response.json()
    files = data.get("files", [])
    selected = [f for f in files if not f.get("private") and
                (f["name"].lower().endswith(".iso") if ONLY_ISO else
                 f.get("source") == "original" and f.get("format") not in
                 {"Metadata", "Archive BitTorrent"} and not f["name"].endswith(".torrent"))]
    if not selected:
        raise ValueError("No public files match the selection.")
    for f in selected:
        safe_path(WORK / item / "verified", f["name"])
        if not re.fullmatch(r"[a-fA-F0-9]{40}", f.get("sha1", "")):
            raise ValueError(f"Missing SHA-1 checksum for {f['name']}.")
    torrents = [f for f in files if f["name"].endswith(".torrent") and not f.get("private")]
    return selected, torrents


def verified(path, info):
    if not path.is_file() or path.stat().st_size != int(info["size"]):
        return False
    digest = hashlib.sha1()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest().lower() == info["sha1"].lower()


def aria(args, timeout=None):
    command = ["aria2c", "--no-conf=true", "--continue=true", "--auto-file-renaming=false",
               "--allow-overwrite=true", "--check-integrity=true", "--file-allocation=none",
               "--summary-interval=10", "--console-log-level=warn", "--max-tries=5",
               "--retry-wait=10", "--connect-timeout=30", "--timeout=60"] + args
    # Inherit stdout to show progress and stop the process if the cell is interrupted.
    process = subprocess.Popen(command)
    try:
        code = process.wait(timeout=timeout)
        if code:
            raise RuntimeError(f"aria2 exited with code {code}")
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()


def torrent_download(item, selected, torrent):
    response = http.get(file_url(item, torrent["name"]), timeout=(15, 90))
    response.raise_for_status()
    meta = bencodepy.decode(response.content)
    info = meta[b"info"]
    root = info[b"name"].decode("utf-8")
    if "/" in root or root in {".", ".."}:
        raise ValueError("Invalid torrent root name.")
    folder = WORK / item / "torrent"
    safe_path(folder, root)
    entries = info.get(b"files")
    mapping = {}
    for index, entry in enumerate(entries if entries is not None else [info], 1):
        if b"l" in entry.get(b"attr", b"") or b"symlink path" in entry:
            raise ValueError("Torrents containing symbolic links are not supported.")
        components = [p.decode("utf-8") for p in entry[b"path"]] if entries is not None else [root]
        if any(p in {"", ".", ".."} or "/" in p or "\\" in p for p in components):
            raise ValueError("Invalid torrent path.")
        name = "/".join(components)
        local = safe_path(folder, root + "/" + name if entries is not None else root)
        mapping[name] = (index, local)
    if any(f["name"] not in mapping for f in selected):
        raise ValueError("The torrent does not include all selected files.")
    folder.mkdir(parents=True, exist_ok=True)
    torrent_file = WORK / item / "source.torrent"
    torrent_file.write_bytes(response.content)
    indices = ",".join(str(mapping[f["name"]][0]) for f in selected)
    aria([f"--dir={folder}", f"--select-file={indices}", "--seed-time=0",
          "--bt-stop-timeout=120", str(torrent_file)], timeout=900)
    for f in selected:
        source = mapping[f["name"]][1]
        if not verified(source, f):
            raise ValueError(f"Verification failed: {f['name']}")
    for f in selected:
        target = safe_path(WORK / item / "verified", f["name"])
        target.parent.mkdir(parents=True, exist_ok=True)
        os.replace(mapping[f["name"]][1], target)


def https_download(item, selected):
    for f in selected:
        target = safe_path(WORK / item / "verified", f["name"])
        if verified(target, f):
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        aria([f"--dir={target.parent}", f"--out={target.name}", "--split=4",
              "--max-connection-per-server=4", f"--checksum=sha-1={f['sha1']}",
              file_url(item, f["name"])])
        if not verified(target, f):
            raise ValueError(f"Verification failed: {f['name']}")

In [ ]:
completed = []
failures = []
for value in inputs:
    try:
        item = identifier(value)
        selected, torrents = get_plan(item)
        pending = [f for f in selected if not verified(safe_path(WORK / item / "verified", f["name"]), f)]
        total = sum(int(f["size"]) for f in pending)
        print(f"\n{item}: {len(selected)} files; pending {total / 1024**3:.2f} GiB", flush=True)
        for f in selected:
            print(f"  {f['name']} ({int(f['size']) / 1024**3:.2f} GiB)")
        # Reserve space for up to two copies: partial torrent and HTTPS fallback.
        if total * 2 + 512 * 1024**2 > shutil.disk_usage(WORK).free:
            raise RuntimeError("Insufficient temporary space (including the HTTPS fallback reserve).")
        if pending and METHOD == "auto" and torrents:
            try:
                print("Downloading torrent (limit: 15 minutes)...", flush=True)
                torrent_download(item, pending, torrents[0])
            except Exception as error:
                print(f"Torrent incomplete: {error}. Trying HTTPS...", flush=True)
                https_download(item, pending)
        elif pending:
            print("Downloading over HTTPS...", flush=True)
            https_download(item, pending)
        completed.extend((item, f) for f in selected)
        print("Complete and verified. Run the final cell to save to Drive.", flush=True)
    except Exception as error:
        failures.append((value, str(error)))
        print(f"ERROR for {value}: {error}", flush=True)
        # Keep individually completed files in the manifest.
        if "selected" in locals() and "item" in locals():
            for f in selected:
                if verified(safe_path(WORK / item / "verified", f["name"]), f):
                    completed.append((item, f))
    finally:
        selected = []
if failures:
    print("\nSome items failed. You can retry in this runtime or save completed files.")

In [ ]:
# Copy only complete files; skip aria2 temporary files.
if not completed:
    raise RuntimeError("No completed files to copy. Run the download first.")
seen = set()
for item, info in completed:
    key = (item, info["name"])
    if key in seen:
        continue
    seen.add(key)
    source = safe_path(WORK / item / "verified", info["name"])
    if not verified(source, info):
        raise RuntimeError(f"The local file no longer passes verification: {source}")
    target = safe_path(DEST / item, info["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if verified(target, info):
        print(f"Already saved: {target}")
        continue
    partial = target.with_name(target.name + ".copying")
    print(f"Copying to Drive: {target}", flush=True)
    shutil.copyfile(source, partial)
    if not verified(partial, info):
        raise RuntimeError(f"The Drive copy fails SHA-1 verification: {partial}")
    os.replace(partial, target)
    print(f"Saved and verified: {target}", flush=True)
print(f"\nDestination: {DEST}")